# Hands-on LLM Fine-Tuning with LoRA
### Faculty Development Program — Session 2

This notebook walks through a complete, minimal LoRA fine-tuning pipeline:

`Environment setup -> Dataset -> Baseline (before) -> LoRA config -> SFT training -> Save/reload adapter -> Fine-tuned (after) -> Compare`

**Base model:** `Qwen/Qwen2.5-0.5B-Instruct` — small enough to fine-tune on a single free-tier
Colab GPU in a workshop setting.

**Runtime note:** This notebook is built for a **GPU runtime** (`Runtime -> Change runtime type -> T4 GPU`
on the free Colab tier). The `peft` / `trl` / `transformers` fine-tuning stack used here targets **CUDA**,
not TPU — a TPU runtime would need a different code path (`torch_xla`) and is out of scope for this
workshop. If your Colab session was assigned a TPU by default, switch it to a T4 GPU before continuing.

**Dataset:** ~440 supervised fine-tuning examples teaching the model to always answer Deep Learning
questions in a strict **Definition / How it works / Why it matters / Example** structure. The point of
this exercise is less about teaching the model *new facts* (the base model already knows most of these
concepts) and more about teaching it a **consistent, reliable response style** — which is exactly the kind
of behavior change fine-tuning is good for (see Session 1, "Why Fine-Tuning?"). This makes the
before/after contrast very easy to see later in the notebook.


## Step 0 — Check GPU & PyTorch

Before anything else, confirm PyTorch can see a CUDA GPU. If `CUDA available` prints `False`,
stop here and switch the Colab runtime to a GPU (`Runtime -> Change runtime type -> T4 GPU`) —
everything downstream will be far too slow on CPU.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## Step 1 — Install Dependencies

We need four libraries beyond base PyTorch:

| Library | Purpose |
|---|---|
| `transformers` | Loads the base model + tokenizer, defines the model architecture |
| `datasets` | Loads and splits our JSON dataset |
| `peft` | Implements LoRA (Parameter-Efficient Fine-Tuning) |
| `trl` | Provides `SFTTrainer`, a Trainer wrapper built for supervised fine-tuning |
| `accelerate` | Handles device placement / mixed precision under the hood |

`-q` keeps the install output quiet; `-U` upgrades to the latest compatible versions.


In [ ]:
!pip install -q -U \
    "torchao>=0.16.0" \
    "peft>=0.17.0" \
    "transformers>=4.45.0" \
    "trl>=0.20.0" \
    "accelerate>=1.0.0"

## Step 2 — Verify Installed Versions

Printing versions here means if something breaks later, we can immediately see whether it's a
version-mismatch issue rather than a bug in our own code.


In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import accelerate

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.15.0
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.10.0
Accelerate: 1.14.0


## Step 3 — Choose the Base Model

`Qwen/Qwen2.5-0.5B-Instruct` is a small (0.5-billion parameter), already instruction-tuned model.
It is:
- Small enough to fine-tune on a free Colab T4 GPU within a workshop session.
- Already instruction-tuned, so it can follow basic instructions out of the box — we are
  layering a *specific, consistent answer format* on top of that existing ability, not teaching
  it to follow instructions from scratch.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Model:", MODEL_NAME)

Model: Qwen/Qwen2.5-0.5B-Instruct


## Step 4 — Load the Fine-Tuning Dataset

We load a pre-generated JSON file, `deep_learning_sft_dataset_v2.json`, containing **~440 examples**.
Upload this file to the same Colab working directory before running this cell
(the file panel on the left, or `files.upload()` if working locally).

**Every example follows the same structure your Session 1 slides described:**

```json
{
  "messages": [
    {"role": "system",    "content": "You are a Deep Learning teaching assistant..."},
    {"role": "user",      "content": "What is backpropagation?"},
    {"role": "assistant", "content": "Definition: ...\nHow it works: ...\nWhy it matters: ...\nExample: ..."}
  ]
}
```

**Why this dataset is bigger than a typical toy demo (~440 vs ~150 examples):** more examples means
more optimizer steps, which means training will take a realistic **10-15+ minutes** on a free T4 GPU —
long enough to actually watch the loss curve move during the workshop, instead of finishing before
everyone has finished reading the code. See the **Step 7b** timing-estimate cell below for a live
projection once you know your assigned GPU's speed.

**Why every answer follows one strict format:** it creates an obvious, easy-to-see behavioral
difference between the *base* model (which answers however it likes) and the *fine-tuned* model
(which should reliably follow the Definition/How it works/Why it matters/Example structure) —
this is the "before vs. after" story of the whole workshop.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

# load_dataset("json", ...) reads our file and wraps it in a HuggingFace Dataset object.
# This mirrors exactly the pattern shown in the Session 1 slides.
dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/FDP/deep_learning_sft_dataset_v2.json"
)["train"]  # load_dataset always nests results under a "train" split key by default

print(dataset)
print("Number of examples:", len(dataset))

print("\nFirst example:")
for message in dataset[0]["messages"]:
    print(f"{message['role'].upper()}:")
    print(message["content"])
    print()


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 444
})
Number of examples: 444

First example:
SYSTEM:
You are a Deep Learning teaching assistant. Always answer using this exact four-part structure, with each label on its own line:
Definition: <one crisp sentence>
How it works: <1-2 sentences on the mechanism>
Why it matters: <1 sentence on practical relevance>
Example: <one short, concrete example>
Be accurate, concise, and use correct technical terminology.

USER:
Can you explain what is causal masking in a decoder-only Transformer?

ASSISTANT:
Definition: Causal masking is a technique that prevents each token's attention from seeing future tokens in the sequence.
How it works: It sets the attention scores for future positions to negative infinity before the softmax, forcing them to zero.
Why it matters: It is essential for autoregressive generation, since the model must not use tokens it hasn't generated yet.
Example: When predicting the 5th token, causal masking hides tokens 6 

## Step 5 — Dataset Sanity Check

Before spending any GPU time, quickly verify the dataset actually looks right: correct roles,
non-empty content, and a sensible number of examples. This is the "checkpoint" step from the
Session 1 slides — never proceed to training on unverified data.


In [ ]:
roles_seen = set()
empty_content_count = 0

for ex in dataset:
    for m in ex["messages"]:
        roles_seen.add(m["role"])
        if not m["content"].strip():
            empty_content_count += 1

print("Roles seen across dataset:", roles_seen)
print("Examples with any empty message content:", empty_content_count)
print("Total examples:", len(dataset))

assert roles_seen == {"system", "user", "assistant"}, "Unexpected roles found — check the dataset."
assert empty_content_count == 0, "Found empty message content — check the dataset."
print("\nDataset structure looks correct.")

from datasets import Dataset

dataset = Dataset.from_list(data)

print(dataset)
print("Number of examples:", len(dataset))

print("\nFirst example:")
print(dataset[0])
for message in dataset[0]["messages"]:
    print(f"{message['role'].upper()}:")
    print(message["content"])
    print()


Roles seen across dataset: {'assistant', 'user', 'system'}
Examples with any empty message content: 0
Total examples: 444

Dataset structure looks correct.
Dataset({
    features: ['messages'],
    num_rows: 156
})
Number of examples: 156

First example:
{'messages': [{'role': 'system', 'content': 'You are a Deep Learning teaching assistant. Explain concepts accurately, clearly, and concisely. Use appropriate technical terminology, correct misconceptions, and give practical intuition when useful. Prefer a short definition followed by the key mechanism or implication.'}, {'role': 'user', 'content': 'What is deep learning?'}, {'role': 'assistant', 'content': 'Deep learning is a branch of machine learning that uses neural networks with multiple learned layers to automatically learn representations from data. The learned representations can support tasks such as classification, prediction, generation, and perception.'}]}
SYSTEM:
You are a Deep Learning teaching assistant. Explain concepts 

## Step 6 — Train / Validation Split

We hold out 15% of examples as a validation set. Validation examples are **never** used to update
model parameters — they exist purely to check whether the model is generalizing rather than just
memorizing the training set (Session 1: "Training vs Validation").

`seed=42` makes the split reproducible — re-running this cell always produces the same split.


In [ ]:
split_dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))

Training examples: 124
Validation examples: 32


## Step 7 — Load the Tokenizer

The tokenizer converts human-readable text into the token IDs the model actually consumes.

`Text -> Tokens -> Token IDs -> Embeddings` (the embeddings step happens inside the model itself).


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully.
Vocabulary size: 151643


In [ ]:
text = "What is backpropagation in deep learning?"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Original text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Original text:
What is backpropagation in deep learning?

Tokens:
['What', 'Ġis', 'Ġback', 'prop', 'agation', 'Ġin', 'Ġdeep', 'Ġlearning', '?']

Token IDs:
[3838, 374, 1182, 2674, 27137, 304, 5538, 6832, 30]


In [ ]:
# The chat template turns a structured {role, content} conversation into the exact
# text format Qwen2.5-Instruct expects, including its special turn-boundary tokens.
# add_generation_prompt=True appends the "assistant, your turn" marker, which is what
# we want when we're about to GENERATE a reply (as opposed to formatting a training example
# that already includes the target response).
messages = [
    {
        "role": "system",
        "content": "You are a Deep Learning teaching assistant."
    },
    {
        "role": "user",
        "content": "What is backpropagation?"
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted_prompt)

<|im_start|>system
You are a Deep Learning teaching assistant.<|im_end|>
<|im_start|>user
What is backpropagation?<|im_end|>
<|im_start|>assistant



## Step 8 — Load the Base LLM

We load `Qwen2.5-0.5B-Instruct` in `float16` to roughly halve memory use versus full `float32`
precision. **At this point, no fine-tuning has happened yet** — every response we get from `model`
below is purely from pretraining + the base instruction-tuning Qwen already shipped with.


In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully.")
print("Model device:", model.device)
print("Model parameters:", model.num_parameters())

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0
Model parameters: 494032768


## Step 9 — Generate Baseline ("Before Fine-Tuning") Responses

We define one `generate_response` helper we will reuse for **both** the base model and, later, the
fine-tuned model — using the exact same function for both is what makes the before/after comparison
fair (Session 1: "Base Model vs Fine-Tuned Model" — same prompts, same generation settings, only the
weights differ).


In [ ]:
def generate_response(
    model,
    tokenizer,
    user_prompt,
    max_new_tokens=150,
    temperature=0.7
):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Deep Learning teaching assistant. "
                "Explain concepts clearly and accurately."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Move input tensors to the same device as the model
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # Number of tokens in the input prompt
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Keep only newly generated tokens
    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
# Fixed evaluation prompts — reused for BOTH the base and fine-tuned model later.
# Keeping this list fixed is what makes the final comparison meaningful.
baseline_prompts = [
    "What is backpropagation?",
    "Why does a neural network need activation functions?",
    "What is the difference between CNNs and Transformers?",
    "Explain overfitting and how dropout helps.",
    "What is the role of the learning rate during training?"
]

In [ ]:
# Generate and print each baseline response as we go, so we can watch progress.
for i, prompt in enumerate(baseline_prompts, 1):
    print(f"\n{'='*70}")
    print(f"Question {i}: {prompt}")
    print(f"{'='*70}")

    response = generate_response(
        model,
        tokenizer,
        prompt
    )

    print(response)

In [ ]:
# Also store results in a dict, keyed by prompt, so we can compare against
# the fine-tuned model's answers later using the exact same prompts.
baseline_results = {}

for prompt in baseline_prompts:
    baseline_results[prompt] = generate_response(model, tokenizer, prompt)

print("Baseline responses captured for", len(baseline_results), "prompts.")
print("Notice: the base model answers correctly, but with NO consistent structure —")
print("that inconsistency is exactly what fine-tuning will fix.")

## Step 10 — Configure LoRA

We do **not** want to update all ~0.5B parameters of the base model. Instead we use
Parameter-Efficient Fine-Tuning (PEFT) via LoRA:

- The pretrained model stays **frozen**.
- Small trainable low-rank adapter matrices are inserted into the attention layers.
- Only those adapter parameters are updated during training.

```
Base Model  -> frozen
LoRA Adapter -> trainable
```

| Argument | Meaning |
|---|---|
| `r=8` | Rank of the low-rank update — controls adapter capacity |
| `lora_alpha=16` | Scales the adapter's contribution (commonly ~2×`r`) |
| `lora_dropout=0.05` | Regularizes the adapter to reduce overfitting on a small dataset |
| `target_modules` | Which layers get adapters — here, the attention projections |


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print(lora_config)


In [ ]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

In [ ]:
# Sanity check: confirm only a small fraction of the model is actually trainable.
model.print_trainable_parameters()

In [ ]:
# Inspect exactly which tensors will receive gradient updates.
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

## Step 11 — Configure Supervised Fine-Tuning (SFT)

Supervised Fine-Tuning trains the model on `input -> desired response` examples. Our dataset is:

`User Question -> Structured Deep Learning Explanation`

The `SFTTrainer` will use these ~440 examples to optimize the small set of trainable LoRA
parameters (the base model stays frozen throughout).

**Sizing these settings for a ~10-15 minute training run on a free T4 GPU:**
- `num_train_epochs=3` — enough passes over ~440 examples to make the style shift stick,
  without training so long that it overfits this fairly small dataset.
- `per_device_train_batch_size=4` with `gradient_accumulation_steps=2` gives an
  **effective batch size of 8** — big enough for stable gradients, small enough to fit
  comfortably in a T4's 16GB of memory alongside the frozen base model.
- `eval_steps` / `save_steps` are set relatively frequently (every ~10-12 steps). Besides
  giving a fine-grained loss curve to watch live, each evaluation pass and each checkpoint
  save is itself real (I/O-bound) wall-clock time — this is one of the honest reasons a
  workshop-scale fine-tune takes noticeably longer than raw compute alone would suggest.


In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./deep_learning_lora",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=10,

    save_strategy="steps",
    save_steps=10,

    fp16=True,

    report_to="none"
)

print(sft_config)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=F

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
print("SFTTrainer created successfully.")

## Step 12 — Pre-Flight Summary & Training Time Estimate

Before starting a multi-minute training run, print out exactly what is about to happen: how many
examples, how many total optimizer steps, and a rough wall-clock estimate. GPU speed varies across
Colab's free-tier hardware (T4 vs. the occasionally-assigned L4/A100), so treat the estimate as a
ballpark, not a guarantee — the actual measured duration is printed after training completes in
Step 13.

**If your actual run comes in well under 10 minutes:** increase `num_train_epochs` (e.g. to 4-5) in
Step 11 and re-run from there — more epochs over the same ~440 examples is the simplest lever.
**If it runs much longer than 15-20 minutes:** reduce `num_train_epochs` back down, or increase
`per_device_train_batch_size` to process more examples per step.


In [ ]:
import math

effective_batch_size = (
    sft_config.per_device_train_batch_size
    * sft_config.gradient_accumulation_steps
)

steps_per_epoch = math.ceil(
    len(train_dataset) / effective_batch_size
)

total_steps = steps_per_epoch * sft_config.num_train_epochs
num_evals = total_steps // sft_config.eval_steps
num_saves = total_steps // sft_config.save_steps

print("Training examples:      ", len(train_dataset))
print("Validation examples:    ", len(eval_dataset))
print("Effective batch size:   ", effective_batch_size)
print("Steps per epoch:        ", steps_per_epoch)
print("Epochs:                 ", sft_config.num_train_epochs)
print("Total training steps:   ", total_steps)
print("Evaluation passes:      ", num_evals, "(every", sft_config.eval_steps, "steps)")
print("Checkpoint saves:       ", num_saves, "(every", sft_config.save_steps, "steps)")

# Rough throughput assumption for a 0.5B-parameter LoRA fine-tune on a free-tier T4 GPU,
# including realistic eval/checkpoint overhead (NOT just raw compute FLOPs).
# This is a heuristic for planning purposes only.
SEC_PER_TRAIN_STEP_RANGE = (2.0, 4.5)   # seconds/step, optimistic-to-conservative
SEC_PER_EVAL_PASS_RANGE = (8.0, 20.0)   # seconds per full eval-set pass
SEC_PER_SAVE_RANGE = (3.0, 8.0)         # seconds per checkpoint save (disk I/O)

low = (total_steps * SEC_PER_TRAIN_STEP_RANGE[0]
       + num_evals * SEC_PER_EVAL_PASS_RANGE[0]
       + num_saves * SEC_PER_SAVE_RANGE[0])
high = (total_steps * SEC_PER_TRAIN_STEP_RANGE[1]
        + num_evals * SEC_PER_EVAL_PASS_RANGE[1]
        + num_saves * SEC_PER_SAVE_RANGE[1])

print(f"\nRough estimated training time: {low/60:.1f} - {high/60:.1f} minutes")
print("(Heuristic estimate only — actual measured time is reported after Step 13 runs.)")


NameError: name 'sft_config' is not defined

## Step 13 — Train

This is the actual training run. Watch the logged `loss` (training loss) and `eval_loss`
(validation loss) — per Session 1, a healthy run shows both trending downward together.
We wrap the call in a timer so you can see the real measured duration for your specific
GPU allocation, and compare it against the Step 12 estimate.


In [ ]:
import time

start_time = time.time()

training_result = trainer.train()

elapsed_seconds = time.time() - start_time
print(f"\nTraining complete. Elapsed time: {elapsed_seconds/60:.1f} minutes "
      f"({elapsed_seconds:.0f} seconds).")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,3.374805,3.007085,2.696039,8081.000000,0.476758


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,3.374805,3.007085,2.696039,8081.000000,0.476758
20,2.440409,2.330631,2.724768,15793.000000,0.551816
30,2.022616,1.962766,2.307922,23973.000000,0.581055
32,2.022616,1.945145,2.285721,25178.000000,0.589614



Training complete. Elapsed time: 1.1 minutes (66 seconds).


In [ ]:
model.print_trainable_parameters()


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [ ]:
# Full step-by-step log history (loss, eval_loss, learning rate, etc. at each logged step).
trainer.state.log_history


[{'loss': 4.143391418457031,
  'grad_norm': 4.465352535247803,
  'learning_rate': 0.000175,
  'entropy': 2.1154749035835265,
  'num_tokens': 4043.0,
  'mean_token_accuracy': 0.4047531008720398,
  'epoch': 0.3225806451612903,
  'step': 5},
 {'loss': 3.3748050689697267,
  'grad_norm': 2.6025264263153076,
  'learning_rate': 0.00014375,
  'entropy': 2.565558171272278,
  'num_tokens': 8081.0,
  'mean_token_accuracy': 0.4527635857462883,
  'epoch': 0.6451612903225806,
  'step': 10},
 {'eval_loss': 3.007085084915161,
  'eval_runtime': 2.167,
  'eval_samples_per_second': 14.767,
  'eval_steps_per_second': 7.383,
  'eval_entropy': 2.6960387378931046,
  'eval_num_tokens': 8081.0,
  'eval_mean_token_accuracy': 0.4767579361796379,
  'epoch': 0.6451612903225806,
  'step': 10},
 {'loss': 2.8102462768554686,
  'grad_norm': 2.7723798751831055,
  'learning_rate': 0.00011250000000000001,
  'entropy': 2.699247193336487,
  'num_tokens': 12160.0,
  'mean_token_accuracy': 0.481823593378067,
  'epoch': 0.967

## Step 14 — Save the LoRA Adapter

We save **only the trained adapter**, not a full new copy of the ~0.5B-parameter base model.
This is one of the practical benefits of PEFT: adapters are small, fast to save, and easy to
share or swap independently of the (much larger, unchanged) base model.


In [ ]:
ADAPTER_PATH = "./deep_learning_lora_adapter"

trainer.save_model(ADAPTER_PATH)

print("LoRA adapter saved to:", ADAPTER_PATH)


LoRA adapter saved to: ./deep_learning_lora_adapter


In [ ]:
import os

print("Adapter directory contents:")
for f in os.listdir(ADAPTER_PATH):
    size_kb = os.path.getsize(os.path.join(ADAPTER_PATH, f)) / 1024
    print(f"  {f}  ({size_kb:,.0f} KB)")


Adapter directory contents:
  chat_template.jinja  (2 KB)
  tokenizer_config.json  (1 KB)
  README.md  (5 KB)
  training_args.bin  (6 KB)
  adapter_config.json  (1 KB)
  tokenizer.json  (11,154 KB)
  adapter_model.safetensors  (4,248 KB)


## Step 15 — Reload the Base Model and Attach the Adapter

To prove the adapter genuinely works standalone (and to mirror how you'd deploy it), we reload a
**fresh copy** of the base model from scratch and attach the saved adapter to it — rather than just
reusing the in-memory `model` object we already trained.


In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

finetuned_model.eval()  # inference mode: disables dropout, no gradient tracking needed

print("Fine-tuned LoRA model loaded successfully.")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Fine-tuned LoRA model loaded successfully.


## Step 16 — Generate "After Fine-Tuning" Responses

We reuse the exact same `generate_response` function and the exact same `baseline_prompts` list
from Step 9 — same prompts, same generation settings, only the weights are different. This is
what makes the comparison fair.


In [ ]:
finetuned_results = {}

for prompt in baseline_prompts:
    finetuned_results[prompt] = generate_response(finetuned_model, tokenizer, prompt)

print("Fine-tuned responses captured for", len(finetuned_results), "prompts.")


Fine-tuned responses captured for 5 prompts.


## Step 17 — Compare Base vs. Fine-Tuned Responses

The goal here is **not** "is the fine-tuned answer different" — a different answer is not
automatically a better answer (Session 1, Slide 41). Look specifically for:

- Does the fine-tuned model consistently follow the **Definition / How it works / Why it
  matters / Example** structure, where the base model did not?
- Is the fine-tuned answer still factually correct?
- Is it more concise and consistent in tone across different questions?


In [ ]:
for prompt in baseline_prompts:
    print("\n" + "=" * 80)
    print("QUESTION:")
    print(prompt)

    print("\nBASE MODEL (before fine-tuning):")
    print(baseline_results[prompt])

    print("\nFINE-TUNED MODEL (after LoRA + SFT):")
    print(finetuned_results[prompt])



QUESTION:
What is backpropagation?

BASE MODEL (before fine-tuning):
Backpropagation is an algorithm used in machine learning to train neural networks by iteratively adjusting the weights of the network based on their errors during training. It's particularly useful for large, complex models where computing the full error can be computationally expensive.

The process works as follows:

1. **Forward Propagation**: In this step, we compute the output of the model (the predicted value) given the input data. This involves passing each neuron through its activation function until it reaches the final layer.

2. **Error Calculation**: The error between the actual output and the expected output is calculated for each neuron. This error reflects how well the model has learned from the training data.

3. **Weight Update**: Based on the computed error, the weight

FINE-TUNED MODEL (after LoRA + SFT):
Backpropagation is an algorithm for training neural networks by propagating error through the 

---
## Wrap-up

You have now run a complete, minimal LoRA fine-tuning pipeline:

1. Loaded a base instruction-tuned LLM and generated **baseline** responses.
2. Prepared and verified a ~440-example supervised fine-tuning dataset.
3. Configured LoRA to train a small adapter instead of the full model.
4. Ran supervised fine-tuning and watched training/validation loss.
5. Saved and reloaded the adapter independently of the base model.
6. Compared base vs. fine-tuned outputs on identical prompts.

**Key takeaway (Session 2):** Don't just fine-tune a model — fine-tune it for a reason, and
measure whether it actually worked.
